In [2]:
import requests
from requests.auth import HTTPBasicAuth
from io import BytesIO
import os
import pandas as pd
import matplotlib.pyplot as plt
import textwrap
import random
from IPython.display import clear_output

ROOT = ".." # Adjust to repository root

from dotenv import load_dotenv
DOTENV_PATH = os.path.join(ROOT,"../../apis/.env") # Adjust to .env file location

if load_dotenv(DOTENV_PATH):
    abs_path = os.path.abspath(DOTENV_PATH)
    drive, rel = os.path.splitdrive(abs_path)
    parts = rel.strip(os.sep).split(os.sep)
    masked = parts.copy()
    masked_parts = 1  # adjust to mask parts of your path
    max_maskable = max(0, len(parts) - 2)
    for i in range(min(masked_parts, max_maskable)):
        idx = len(parts) - 3 - i
        masked[idx] = "***"
    prefix = f"{drive}{os.sep}" if drive else (os.sep if abs_path.startswith(os.sep) else "")
    print(f"Loaded .env from {prefix}{os.sep.join(masked)}")
else:
    print("Failed to load .env file.")

API_KEY = os.getenv("AEMET_API_KEY")

def mask_token(token, unmasked_chars=3):
    return token[:unmasked_chars] + '*' * (len(token) - unmasked_chars*2) + token[-unmasked_chars:]
print(f"AEMET_API_KEY: {mask_token(API_KEY)}")

Loaded .env from c:\Users\david\***\apis\.env
AEMET_API_KEY: eyJ***********************************************************************************************************************************************************************************************************************************************************************************************************rF4


In [18]:
BASE_URL = "https://opendata.aemet.es/opendata/api"
HEADERS = {"api_key": API_KEY}

def _get_data_url(endpoint: str, params: dict | None = None) -> str | None:
    resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params)
    resp.raise_for_status()
    payload = resp.json()
    return payload.get("datos")


def _download_json(url: str):
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()


Current observations for all stations

In [ ]:
datos_url = _get_data_url("observacion/convencional/todas")
observations = _download_json(datos_url)

print(len(observations))
print(observations[0])

9398
{'idema': '0009X', 'lon': 0.963335, 'fint': '2026-01-27T22:00:00+0000', 'prec': 0.0, 'alt': 406.0, 'vmax': 14.8, 'vv': 7.4, 'dv': 266.0, 'lat': 41.213892, 'dmax': 270.0, 'ubi': 'ALFORJA', 'hr': 82.0, 'tamin': 7.1, 'ta': 7.2, 'tamax': 7.2}


In [30]:
random_point = random.choice(observations)
print(random_point)

{'idema': '9427X', 'lon': -1.380829, 'fint': '2026-01-28T07:00:00+0000', 'prec': 0.0, 'alt': 370.0, 'vmax': 6.1, 'vv': 1.7, 'dv': 86.0, 'lat': 41.481115, 'dmax': 265.0, 'ubi': 'LA ALMUNIA DE DOÑA GODINA', 'hr': 70.0, 'tamin': 5.6, 'ta': 5.6, 'tamax': 6.3}


Daily climatological values for a station

In [31]:
def current_observations_all_stations():
    meta = requests.get(
        f"{BASE_URL}/observacion/convencional/todas",
        headers=HEADERS,
    ).json()

    datos_url = meta["datos"]
    return requests.get(datos_url).json()


data = current_observations_all_stations()

print(len(data))
print(data[0])

9398
{'idema': '0009X', 'lon': 0.963335, 'fint': '2026-01-27T22:00:00+0000', 'prec': 0.0, 'alt': 406.0, 'vmax': 14.8, 'vv': 7.4, 'dv': 266.0, 'lat': 41.213892, 'dmax': 270.0, 'ubi': 'ALFORJA', 'hr': 82.0, 'tamin': 7.1, 'ta': 7.2, 'tamax': 7.2}


Forecasts:

In [32]:
municipality_code = "28079"  # Madrid

endpoint = f"prediccion/especifica/municipio/diaria/{municipality_code}"

datos_url = _get_data_url(endpoint)
forecast = _download_json(datos_url)

print(forecast[0]["prediccion"]["dia"][0])

{'probPrecipitacion': [{'value': 0, 'periodo': '00-24'}, {'value': 0, 'periodo': '00-12'}, {'value': 95, 'periodo': '12-24'}, {'value': 0, 'periodo': '00-06'}, {'value': 100, 'periodo': '06-12'}, {'value': 0, 'periodo': '12-18'}, {'value': 90, 'periodo': '18-24'}], 'cotaNieveProv': [{'value': '', 'periodo': '00-24'}, {'value': '', 'periodo': '00-12'}, {'value': '1200', 'periodo': '12-24'}, {'value': '', 'periodo': '00-06'}, {'value': '1000', 'periodo': '06-12'}, {'value': '', 'periodo': '12-18'}, {'value': '1300', 'periodo': '18-24'}], 'estadoCielo': [{'value': '', 'periodo': '00-24', 'descripcion': ''}, {'value': '', 'periodo': '00-12', 'descripcion': ''}, {'value': '23', 'periodo': '12-24', 'descripcion': 'Intervalos nubosos con lluvia'}, {'value': '', 'periodo': '00-06', 'descripcion': ''}, {'value': '25', 'periodo': '06-12', 'descripcion': 'Muy nuboso con lluvia'}, {'value': '17', 'periodo': '12-18', 'descripcion': 'Nubes altas'}, {'value': '45n', 'periodo': '18-24', 'descripcion':